In [1680]:
import numpy as np
from IPython.display import display, Math, Latex

In [1681]:
EPSILON = 1e-10


def isSame(x1, x2, epsilon=EPSILON):
    return np.abs(x1 - x2) < epsilon

# Методы решений

## Половинное деление

Пусть известно, что на отрезке $ [a, b] $ у функции $ f(x) $ есть один корень. Это означает, что $ f(a) f(b) < 0 $.

Будем рассматривать точку $ t = \frac{1}{2} (a + b) $ - середину отрезка. Тогда если $ f(t) f(a) < 0 $, то $ f(t) f(a) > 0 $ и корень находится на отрезке $ [a, t] $. Аналогично в обратную сторону.

Таким образом, для решения уравнения достаточно найти некоторый отрезок, на котором есть единственный корень.

In [1682]:
def solveBisection(f, a, b, epsilon=EPSILON):
    while True:
        t = (a + b) / 2
        if f(a) * f(t) < 0:
            b = t
        elif f(a) * f(t) > 0:
            a = t
        else:
            return t

        if isSame(a, b, epsilon):
            return (a + b) / 2
        

## Метод простых итераций

Пусть уравнение $ f(x) = 0 $ представимо в виде $ x = \varphi(x) $ на некотором отрезке, включающем корень.

Пусть при этом $ |\varphi'(x)| < 1 $ на этом отрезке.

Пусть $ x^* $ - корень уравнения. Тогда $ x^* = \varphi(x^*) $

Будем рассматривать итеративный процесс: $ x_{k+1} = \varphi(x_k)$

Рассмотрим после очередной итерации:

$ |x_{k + 1} - x^*| = |\varphi(x_k) - x^*| = |\varphi(x_k) - \varphi(x^*)| $

По теореме Лагранжа существует точка $ \xi_k $, принадлежащая интервалу $ (x^*, x_k) $, такая что:

$ \varphi(x_k) - \varphi(x^*) = \varphi'(\xi_k) (x_k - x^*) $

Обозначим $ \alpha $ - максимальное значение $ |\varphi'(x)| $ на исходном отрезке.

Тогда:

$ |x_{k + 1} - x^*| < \alpha |x_k - x^*| $.

Таким образом: $ |x_k - x^*| < \alpha^k |x_0 - x^*| $, значит $x_k$ сходится к $x^*$

In [1683]:
def solveSimpleIterations(phi, x0, epsilon=EPSILON):
    x = x0
    while True:
        x_next = phi(x)
        if isSame(x, x_next, epsilon):
            return x_next
        x = x_next

## Метод Ньютона

Формула Ньютона: $ x_{k + 1} = x_k - \frac{f(x_k)}{f'(x_k)}$.

Пусть есть отрезок, содержащий единственный ноль первого порядка. Без ограничения общности считаем, что в этом отрезке $ |f'(x)| > m > 0 $. Считаем, что $ x^* $ - корень уравнения $ f(x) = 0 $.

Разложим $ f(x) $ по формуле Тейлора в форму Лагранжа в точке $ x_k $:

$ f(x^*) = f(x_k) + f'(x_k)(x^* - x_k) + \frac{f''(\xi_k)}{2}(x^* - x_k)^2 $

Тогда:

$ 0 = f(x_k) - f'(x_k) e_n + \frac{f''(\xi_k)}{2}e_n^2 $.

Отсюда получаем $ f(x_k) = f'(x_k)e_n - \frac{f''(\xi_k)}{2}e_n^2 $.

$ e_{k + 1} = x_{x+1} - x^* = x_k - \frac{f(x_k)}{f'(x_k)} = \frac{f''(\xi_k)}{2f'(x_k)}e_n^2 $ - сходится квадратично.

Для решения достаточно подобрать начальное приближение так, чтобы $ |\frac{f''(x)}{2f'(x)}e_0^2| < 1 $ на некотором отрезке.

In [1684]:
def solveNewton(f, f_der, x0, epsilon=EPSILON):
    x = x0
    while True:
        x_next = x - f(x) / f_der(x)
        if isSame(x, x_next, epsilon):
            return x_next
        x = x_next

## Модифицированный метод Ньютона

То же самое, что метод Ньютона, но в предположении, что $ f'(x) $ слабо меняется на рассматриваемом отрезке. Тогда можно считать:

$ f'(x_k) \approx f'(x_0) $

In [1685]:
def solveModifiedNewton(f, f_der, x0, epsilon=EPSILON):
    x = x0
    f_der_value_x0 = f_der(x0)
    while True:
        x_next = x - f(x) / f_der_value_x0
        if isSame(x, x_next, epsilon):
            return x_next
        x = x_next

# Пункт и)
$ f(x) = x^2 - \frac{e^x}{5} = 0 $

In [1686]:
def f(x):
    return x**2 - np.exp(x) / 5

## Нахождение отрезков, содержащих корни

In [1687]:
values = np.array([-1, 0, 1, 4, 5])
f_values = f(values)
for i in range(len(values)):
    display(Math(rf"f({values[i]}) \approx {f_values[i] : .3f}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Таким образом, есть корни на интервалах $ (-1, 0) $, $ (0, 1) $ и $ (4, 5) $

## Покажем что других корней нет

$ f'(x) = 2x - \frac{e^x}{5} $

$ f''(x) = 2 - \frac{e^x}{5} $

$ f'''(x) = - \frac{e^x}{5} $

Получается, что $ \forall x $: $ f'''(x) < 0 $

Пусть есть хотя бы 4 различных корня: $x_1$, $x_2$, $x_3$, $x_4$

По теореме Ролля существует 3 корня для $f'(x)$. Далее 2 корня для $f''(x)$ и далее 1 корень для $f'''(x)$, что неверно. Значит у функции не более 3 различных корней.

Таким образом, $f(x)$ имеет 3 корня на интервалах $ (-1, 0) $, $ (0, 1) $ и $ (4, 5) $

## Половинное деление

In [1688]:
x = np.array([
    solveBisection(f, -1, 0),
    solveBisection(f, 0, 1),
    solveBisection(f, 4, 5)    
])

#
# Половинное деление
#
display(Math(rf"Корни: x_1 = {x[0]: .3f}, x_2 = {x[1]: .3f}, x_3 = {x[2]: .3f}, \epsilon = {EPSILON}"))

<IPython.core.display.Math object>

## Метод простых итераций

### Для сходимости нужно привести уравнение к виду

$ x = \frac{e^{x/2}}{\sqrt{5}} $

$ \varphi(x) = \frac{e^{x/2}}{\sqrt{5}} $

$ \varphi'(x) = \frac{e^{x/2}}{2\sqrt{5}} $ - строго возрастает.

или

$ x = ln(5x^2) = \varphi_1(x) $

$ \varphi_1'(x) = \frac{10x}{5x^2} = \frac{2}{x}$

In [1689]:
def phi_derivative(x):
    return np.exp(x / 2) / (2 * np.sqrt(5))

values = np.array([-1, 0, 1, 4, 5])
for val in values:
    display(Math(rf"\varphi'({val}) = {phi_derivative(val) : .3f}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [1690]:
def phi1_derivative(x):
    return 2 / x

values = np.array([4, 5])

for val in values:
    display(Math(rf"\varphi_1'({val}) = {phi1_derivative(val) : .3f}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Таким образом, подобраны функции, подходящие для метода простых итераций:

$ \varphi_1(x) = - \frac{e^{x/2}}{\sqrt{5}} $ - на $ (-1, 0) $

$ \varphi_2(x) = \frac{e^{x/2}}{\sqrt{5}} $ - на $ (0, 1) $

$ \varphi_3(x) = \ln(5x^2)$ - на $ (4, 5) $

In [1691]:
# [-1, 0]
x0_1 = (-1 + 0) / 2
def phi1(x):
    return -np.exp(x / 2) / np.sqrt(5)

# [0, 1]
x0_2 = (0 + 1) / 2
def phi2(x):
    return np.exp(x / 2) / np.sqrt(5)

# [4, 5]
x0_3 = (4 + 5) / 2
def phi3(x):
    return np.log(5 * (x**2))

x = np.array([
    solveSimpleIterations(phi1, x0_1),
    solveSimpleIterations(phi2, x0_2),
    solveSimpleIterations(phi3, x0_3),    
])

display(Math(rf"Корни: x_1 = {x[0]: .3f}, x_2 = {x[1]: .3f}, x_3 = {x[2]: .3f}, \epsilon = {EPSILON}"))

<IPython.core.display.Math object>

## Метод Ньютона

In [1692]:
def f_der(x):
    return 2 * x - np.exp(x) / 5

x0_1 = (-1 + 0) / 2
x0_2 = (0 + 1) / 2
x0_3 = (4 + 5) / 2
x = np.array([
    solveNewton(f, f_der, x0_1),
    solveNewton(f, f_der, x0_2),
    solveNewton(f, f_der, x0_3)
])

display(Math(rf"Корни: x_1 = {x[0]: .3f}, x_2 = {x[1]: .3f}, x_3 = {x[2]: .3f}, \epsilon = {EPSILON}"))

<IPython.core.display.Math object>

## Модифицированный метод Ньютона

In [1693]:
def f_der(x):
    return 2 * x - np.exp(x) / 5

x0_1 = (-1 + 0) / 2
x0_2 = (0 + 1) / 2
x0_3 = (4 + 5) / 2
x = np.array([
    solveModifiedNewton(f, f_der, x0_1),
    solveModifiedNewton(f, f_der, x0_2),
    solveModifiedNewton(f, f_der, x0_3)
])

display(Math(rf"Корни: x_1 = {x[0]: .3f}, x_2 = {x[1]: .3f}, x_3 = {x[2]: .3f}, \epsilon = {EPSILON}"))

<IPython.core.display.Math object>

# Пункт з)
$ f(x) = 2\lg(x) - \frac{x}{2} + 1 $

In [1694]:
def f(x):
    return 2 * np.log10(x) - x / 2 + 1

## Нахождение оторезков, которые содержат корни

In [1695]:
values = np.array([0.1, 2, 5])
for val in values:
    display(Math(rf"f({val}) = {f(val) : .3f}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Таким образом, у функции точно есть корни на интервалах $ (0.1, 2) $ и $ (2, 5) $

Покажем что других корней быть не может.

$ f'(x) = 2 \frac{1}{x \ln(10)} - \frac{1}{2} $

$ f''(x) = \frac{-2} {x^2 \ln(10)} < 0 $

Пусть есть хотя бы 3 различных корня. Тогда между ними есть 2 корня $ f'(x) $, а между ними 1 корень $ f''(x) $, чего быть не может.

Значит у $ f(x) $ есть 2 корня: на отрезке $ (0.1, 2) $ и на отрезке $ (2, 5) $

## Половинное деление

In [1696]:
x = np.array([
    solveBisection(f, 0.1, 2),
    solveBisection(f, 2, 5),    
])

display(Math(rf"Корни: x_1 = {x[0]: .3f}, x_2 = {x[1]: .3f}, \epsilon = {EPSILON}"))

<IPython.core.display.Math object>

## Метод простой итерации

$ x = 4 \lg(x) + 2 = \varphi(x)$

$ \varphi'(x) = \frac{4}{x \ln{10}} $ - строго убывает

или

$ x = 10^{\frac{x}{4} - \frac{1}{2}} = \varphi_1(x) $

$ \varphi_1'(x) = \frac{1}{4} \ln{10} * 10^{\frac{x}{4} - \frac{1}{2}} $ - строго возрастает

In [1697]:
def phi_der(x):
    return 4 / (x * np.log(10))

def phi1_der(x):
    return 0.25 * np.log(10) * np.pow(10, x/4 - 0.5)

values = [0.1, 2, 5]
for val in values:
    display(Math(rf"\varphi'({val}) = {phi_der(val): .3f}"))
    display(Math(rf"\varphi_1'({val}) = {phi1_der(val): .3f}"))


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Таким образом на $ (0.1, 2) $:

$ |\varphi_1'(x)| < 1 $

На $ (2, 5) $:

$ |\varphi'(x)| < 1 $

In [1698]:
# [2, 5]
x0 = (2 + 5) / 2
def phi(x):
    return 4 * np.log10(x) + 2

x0_1 = (0.1 + 2) / 2
def phi1(x):
    return np.pow(10, x / 4 - 0.5)

def find(phi, a, b, epsilon):
    x = (a + b) / 2
    while True:
        x_next = phi(x)
        if (np.abs(x_next - x) < epsilon):
            return x_next
        x = x_next

x = np.array([
    solveSimpleIterations(phi1, x0_1),
    solveSimpleIterations(phi, x0),
])
display(Math(rf"Корни: x_1 = {x[0]: .3f}, x_2 = {x[1]: .3f}, \epsilon = {EPSILON}"))

<IPython.core.display.Math object>

## Метод Ньютона

$ f'(x) = 2 \frac{1}{x \ln(10)} - \frac{1}{2} $

Подбираем начальные приближения так, чтобы метод сходился.

In [1699]:
def f_der(x):
    return 2 / (x * np.log(10)) - 0.5

x = np.array([
    solveNewton(f, f_der, 0.5),
    solveNewton(f, f_der, 4),
])

display(Math(rf"Корни: x_1 = {x[0]: .3f}, x_2 = {x[1]: .3f}, \epsilon = {EPSILON}"))

<IPython.core.display.Math object>

## Модифицированный метод Ньютона

In [1700]:
x = np.array([
    solveModifiedNewton(f, f_der, 0.5),
    solveModifiedNewton(f, f_der, 4)    
])

display(Math(rf"Корни: x_1 = {x[0]: .3f}, x_2 = {x[1]: .3f}, \epsilon = {EPSILON}"))

<IPython.core.display.Math object>

# Системы

## Задание IV.12.4 а)

$\begin{cases} \sin(x+1) - y = 1.2, \\ 2x + \cos(y) = 2 \end{cases}$

Сводится к уравнению:
$ x = 1 - \frac{1}{2} \cos (\sin(x + 1) - 1.2) $

$ f(x) = x - 1 + \frac{1}{2} \cos (\sin(x + 1) - 1.2) = 0 $

In [1701]:
def f(x):
    return x - 1 + 0.5 * np.cos(np.sin(x + 1) - 1.2)

def calc_y(x):
    return np.sin(x + 1) - 1.2

### Поиск отрезков с корнями

In [1702]:
values = [0, 1]
for val in values:
    display(Math(rf"f({val}) = {f(val) : .3f}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Покажем что других корней нет.

$ f'(x) = 1 - \frac{1}{2} \sin (\sin(x + 1) - 1.2) \cos(x + 1) > 0 $.

Пусть есть хотя бы 2 различных корня. Тогда по теореме Ролля у $ f'(x) $ между ними есть корень - противоречие.

Значит у $ f(x) $ есть единственный корень $x^*$ и он находится на отрезке $ [0, 1] $

### Половинное деление

In [1703]:
x = solveBisection(f, 0, 1)
y = calc_y(x)

display(Math(rf"(x, y) = ({x: .3f}, {y: .3f})"))

<IPython.core.display.Math object>

### Метод простых итераций

$ x = 1 - \frac{1}{2} \cos (\sin(x + 1) - 1.2) = \varphi(x) $

$ \varphi'(x) = 0.5 \sin(\sin(x + 1) - 1.2) \cos(x + 1) $ - строго возрастает на $ [0, 1] $

In [1704]:
def phi_der(x):
    return 0.5 * np.sin(np.sin(x + 1) - 1.2) * np.cos(x + 1)

display(Math(rf"\varphi'(0) = {phi_der(0) : .3f}"))
display(Math(rf"\varphi'(1) = {phi_der(1) : .3f}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [1705]:
def phi(x):
    return 1 - 0.5 * np.cos(np.sin(x + 1) - 1.2)
x0 = 0

x = solveSimpleIterations(phi, x0)
y = calc_y(x)

display(Math(rf"(x, y) = ({x: .3f}, {y: .3f})"))

<IPython.core.display.Math object>

### Метод Ньютона

$ f'(x) = 1 - \frac{1}{2} \sin (\sin(x + 1) - 1.2) \cos(x + 1) > 0 $.

In [1706]:
def f_der(x):
    return 1 - 0.5 * np.sin(np.sin(x + 1) - 1.2) * np.cos(x + 1)

x0 = 0

x = solveNewton(f, f_der, x0)
y = calc_y(x)

display(Math(rf"(x, y) = ({x: .3f}, {y: .3f})"))

<IPython.core.display.Math object>

### Модифицированный метод Ньютона

In [1707]:
x0 = 0

x = solveModifiedNewton(f, f_der, x0)
y = calc_y(x)

display(Math(rf"(x, y) = ({x: .3f}, {y: .3f})"))

<IPython.core.display.Math object>

## 

## Задание IV.12.6 б)

$\begin{cases} 2x^2 - xy - 5x + 1 = 0, \\ x + 3 \lg{x} - y^2 = 0 \end{cases}$

Из первого выражаем $ y $ ($x=0$ - не решение) и подставляем во второе:

$ x + 3\lg{x} - (2x - 5 + \frac{1}{x})^2 = f(x) = 0 $

In [1708]:
def f(x):
    return x + 3 * np.log10(x) - (2 * x - 5 + 1 / x)**2

## Находим отрезки с корнями

In [1709]:
values = [1, 2, 3, 4]

for val in values:
    display(Math(rf"f({val}) = {f(val): .3f}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

Значит у $f(x)$ есть корени на отрезках $[1, 2]$ и $[3, 4]$

Аналогично, поскольку $ f''(x) < 0 $, то корней не может быть больше двух.

## Половинное деление

In [1710]:
x = np.array([
    solveBisection(f, 1, 2),
    solveBisection(f, 3, 4),    
])
y = calc_y(x)

for i in range(len(x)):
    display(Math(rf"(x, y) = ({x[i]:.3f}, {y[i]:.3f})"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Метод простых итераций

Выбираем $ m $, такое, чтобы на $ [1, 2] $ выполнялось $ |1 - mf'(x)| < 1 $, тогда можно будет применить метод простых итераций с функцией $ \varphi_1(x) = 1 - mf(x) $

Аналогично для отрезка $ [1, 2] $

Подходят:

$ \varphi_1(x) = x - 0.1 f(x) $

$ \varphi_2(x) = x + 0.1 f(x) $

In [1711]:
# [1, 2]
def phi1(x):
    return x - 0.1 * f(x)
x0_1 = (1 + 2) / 2

# [3, ]
def phi2(x):
    return x + 0.1 * f(x)
x0_2 = (3 + 4) / 2

x = np.array([
    solveSimpleIterations(phi1, x0_1),
    solveSimpleIterations(phi2, x0_2),    
])
y = calc_y(x)

for i in range(len(x)):
    display(Math(rf"(x, y) = ({x[i]:.3f}, {y[i]:.3f})"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Метод Ньютона

In [1712]:
def f_der(x):
    return 1 + 3 / (x * np.log(10)) - 2 * (2 * x - 5 + 1/x) * (2 - np.pow(x, -2))

x0_1 = (1 + 2) / 2
x0_2 = (3 + 4) / 2

x = np.array([
    solveNewton(f, f_der, x0_1),
    solveNewton(f, f_der, x0_2),    
])
y = calc_y(x)

for i in range(len(x)):
    display(Math(rf"(x, y) = ({x[i]:.3f}, {y[i]:.3f})"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

## Модифицированный метод Ньютона

In [1713]:
x = np.array([
    solveModifiedNewton(f, f_der, x0_1),
    solveModifiedNewton(f, f_der, x0_2),    
])
y = calc_y(x)

for i in range(len(x)):
    display(Math(rf"(x, y) = ({x[i]:.3f}, {y[i]:.3f})"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>